In [8]:
import pandas as pd

# 1⃣  Cargar los CSV (o Parquet) que están en S3 o en disco -------------
clientes = pd.read_csv("s3://s3-mlflow-artifacts-mlflow-tracking-server-grupo5-01ggz/sagemaker/clients-attrition/grupo5/data_train/trained_clientes_clean.csv")
requerimientos = pd.read_csv("s3://s3-mlflow-artifacts-mlflow-tracking-server-grupo5-01ggz/sagemaker/clients-attrition/grupo5/data_train/train_requerimientos_sample.csv")


In [13]:

# ------------------------------------------------------------------
# 3⃣  NORMALIZAR NOMBRES DE COLUMNAS  (minúsculas + sin espacios)
# ------------------------------------------------------------------
def clean_cols(df):
    df.columns = (
        df.columns
          .str.strip()        # quita espacios inicio/fin
          .str.lower()        # todo en minúsculas
    )
    return df

clientes       = clean_cols(clientes)
requerimientos = clean_cols(requerimientos)

# ------------------------------------------------------------------
# 4⃣  RENOMBRAR LAS CLAVES DIFERENTES / CREAR ALIAS
# ------------------------------------------------------------------
clientes = clientes.rename(columns={
    "id_correlativo": "id_cliente",
    "codmes":         "codmes_cliente"
})

requerimientos = requerimientos.rename(columns={
    "codmes": "codmes_requerimiento"
})

# --- conserva solo las columnas que realmente existen -------------
# (puedes borrarlo si quieres incluir TODAS)
keep_clientes = [
    "id_cliente", "codmes_cliente", "flg_bancarizado", "rang_ingreso",
    "flag_lima_provincia", "edad", "antiguedad", "attrition"
]  # añade más si lo necesitas

keep_reqs = [
    "id_correlativo", "codmes_requerimiento",
    "tipo_requerimiento2", "dictamen",
    "producto_servicio_2", "submotivo_2"
]

clientes       = clientes[ [c for c in keep_clientes if c in clientes.columns] ]
requerimientos = requerimientos[ [c for c in keep_reqs    if c in requerimientos.columns] ]

# ------------------------------------------------------------------
# 5⃣  LEFT JOIN  (clientes ⭢ requerimientos)
# ------------------------------------------------------------------
df = (
    clientes
      .merge(
          requerimientos,
          how="left",
          left_on="id_cliente",
          right_on="id_correlativo",
          suffixes=("", "_req")
      )
      .drop(columns=["id_correlativo"])  # ya no hace falta
)

print("Shape final:", df.shape)
print(df.head())

Shape final: (69224, 13)
   id_cliente  codmes_cliente  flg_bancarizado     rang_ingreso  \
0       35653          201208                1  Rang_ingreso_06   
1       66575          201208                1  Rang_ingreso_03   
2       56800          201208                1  Rang_ingreso_01   
3        8410          201208                1  Rang_ingreso_04   
4        6853          201208                1              NaN   

  flag_lima_provincia  edad  antiguedad  attrition  codmes_requerimiento  \
0                Lima  25.0         6.0          0              201208.0   
1           Provincia  27.0         0.0          0                   NaN   
2           Provincia  34.0         4.0          0              202405.0   
3           Provincia  63.0         5.0          0              202404.0   
4                Lima  25.0         0.0          0                   NaN   

  tipo_requerimiento2         dictamen producto_servicio_2    submotivo_2  
0             Reclamo       NO PROCEDE 

In [14]:
# 4⃣  (Opcional) Guardar a CSV o Parquet -----------------------------------
df.to_csv("s3://s3-mlflow-artifacts-mlflow-tracking-server-grupo5-01ggz/sagemaker/clients-attrition/grupo5/data_train/trained_clientes_pipeline.csv", index=False)

In [11]:
print(clientes.columns.tolist())

['ID_CORRELATIVO', 'CODMES', 'FLG_BANCARIZADO', 'RANG_INGRESO', 'FLAG_LIMA_PROVINCIA', 'EDAD', 'ANTIGUEDAD', 'ATTRITION', 'RANG_SDO_PASIVO_MENOS0', 'SDO_ACTIVO_MENOS0', 'SDO_ACTIVO_MENOS1', 'SDO_ACTIVO_MENOS2', 'SDO_ACTIVO_MENOS3', 'SDO_ACTIVO_MENOS4', 'SDO_ACTIVO_MENOS5', 'FLG_SEGURO_MENOS0', 'FLG_SEGURO_MENOS1', 'FLG_SEGURO_MENOS2', 'FLG_SEGURO_MENOS3', 'FLG_SEGURO_MENOS4', 'FLG_SEGURO_MENOS5', 'RANG_NRO_PRODUCTOS_MENOS0', 'FLG_NOMINA', 'NRO_ACCES_CANAL1_MENOS0', 'NRO_ACCES_CANAL1_MENOS1', 'NRO_ACCES_CANAL1_MENOS2', 'NRO_ACCES_CANAL1_MENOS3', 'NRO_ACCES_CANAL1_MENOS4', 'NRO_ACCES_CANAL1_MENOS5', 'NRO_ACCES_CANAL2_MENOS0', 'NRO_ACCES_CANAL2_MENOS1', 'NRO_ACCES_CANAL2_MENOS2', 'NRO_ACCES_CANAL2_MENOS3', 'NRO_ACCES_CANAL2_MENOS4', 'NRO_ACCES_CANAL2_MENOS5', 'NRO_ACCES_CANAL3_MENOS0', 'NRO_ACCES_CANAL3_MENOS1', 'NRO_ACCES_CANAL3_MENOS2', 'NRO_ACCES_CANAL3_MENOS3', 'NRO_ACCES_CANAL3_MENOS4', 'NRO_ACCES_CANAL3_MENOS5', 'NRO_ENTID_SSFF_MENOS0', 'NRO_ENTID_SSFF_MENOS1', 'NRO_ENTID_SSFF_MENOS

In [12]:
missing = set(cols_c.keys()) - set(clientes.columns)
print("Columnas que no se encuentran:", missing)

Columnas que no se encuentran: {'rang_ingreso', 'codmes', 'attrition', 'flag_lima_provincia', 'id_correlativo', 'flg_bancarizado', 'edad', 'antiguedad'}
